# ANALYSE-08 : Ramsey et van der Waerden — l'énumération qui prouve, et le mur où elle s'arrête

Deux théorèmes exacts de
petite combinatoire — $R(3,3) = 6$ et $W(2,3) = 9$ — y sont prouvés **par épuisement
brutal** : énumérer toutes les colorations, compter les survivantes, constater zéro.
Puis le même geste est tendu vers $R(4,4)$, où il meurt sur un mur de $2^{153}$
colorations — et l'argument probabiliste d'Erdős (1947) prend le relais : quand
compter devient impossible, **majorer** suffit pour l'existence.

Règle du carnet, héritée de la série : chaque affirmation numérique est **mesurée par
la cellule qui la précède**, ou citée avec sa source.
## Navigation

| Index | Python | Précédent | Suivant |
|---|---|---|---|
| Index | [ANALYSE-08 (Python)](ANALYSE-08-Ramsey-VdW.ipynb) | [ANALYSE-07 - Percolation critique](ANALYSE-07-Percolation-Critique.ipynb) | [ANALYSE-09 - Tuilage apériodique](ANALYSE-09-Tuilage-Aperiodique.ipynb) |

## Plan du carnet

1. **Section 1** — $R(3,3)$ par épuisement sur $K_4$, $K_5$, $K_6$, et la preuve classique (sans machine) de la borne supérieure.
2. **Section 2** — le pont comptage → existence : l'argument d'Erdős (1947), puis le mur de $R(4,4)$ chiffré.
3. **Section 3** — $W(2,3)$ par le même moteur d'énumération, appliqué à un autre objet.
4. **Section 4** — synthèse mesurée en un seul tableau, puis trois exercices.

**Convention d'encodage.** Une coloration des arêtes de $K_n$ est un tuple de $\binom{n}{2}$ bits — le bit de l'arête $(i,j)$ vaut 1 si elle est rouge, 0 si elle est bleue —, exactement la variable `bits` des cellules de code ; sur $[1,n]$, c'est un tuple de $n$ bits. **Prérequis** : dénombrement (coefficients binomiaux) et probabilités élémentaires (espérance, linéarité).

## 1. Ramsey $R(3,3)$ : l'énoncé, et la preuve par épuisement

**Énoncé.** Colorons chaque arête de $K_n$ (le graphe complet sur $n$ sommets) en
rouge ou bleu. $R(3,3)$ est le plus petit $n$ tel que **toute** coloration contienne un
triangle monochrome. La valeur $R(3,3) = 6$ se démontre en une soirée d'ordinateur :
énumérer les $2^{\binom{n}{2}}$ colorations et vérifier, pour chacune, ses
$\binom{n}{3}$ triangles.

La stratégie du carnet : monter $n = 4, 5, 6$ et regarder le décompte s'effondrer —
18 survivantes, puis 12, puis 0.


La borne supérieure $R(3,3) \le 6$ se démontre aussi à la main, par le principe des tiroirs : dans toute 2-coloration des arêtes de $K_6$, fixons un sommet $v$ ; ses 5 arêtes incidentes ne reçoivent que deux couleurs, donc au moins 3 d'entre elles — disons vers $a$, $b$, $c$ — ont la même couleur, mettons rouge. Si l'une des arêtes $ab$, $ac$, $bc$ est rouge, elle forme avec $v$ un triangle rouge ; sinon ces trois arêtes sont toutes bleues et forment un triangle bleu. Dans tous les cas, $K_6$ contient un triangle monochrome : $R(3,3) \le 6$. L'énumération ci-dessous retrouve ce verdict par la machine — et exhibe en outre les 12 témoins de $K_5$ qui donnent $R(3,3) > 5$.

In [2]:
fn edges_of(n: usize) -> Vec<(usize, usize)> {
    // aretes (i < j) de K_n ; la position dans cette liste indexe la coloration
    let mut out = Vec::new();
    for i in 0..n { for j in i+1..n { out.push((i, j)); } }
    out
}
fn triangles_of(n: usize) -> Vec<(usize, usize, usize)> {
    let mut out = Vec::new();
    for a in 0..n { for b in a+1..n { for c in b+1..n { out.push((a,b,c)); } } }
    out
}
fn for_each_bits<F: FnMut(&[u8])>(m: usize, mut f: F) {
    for mask in 0u32..(1u32 << m) {
        let bits: Vec<u8> = (0..m).rev().map(|j| ((mask >> j) & 1) as u8).collect();
        f(&bits);
    }
}
fn mono_triangle(n: usize, bits: &[u8]) -> Option<(usize, usize, usize)> {
    // premier triangle monochrome de la coloration bits (None si aucune)
    let e = edges_of(n);
    for &(a, b, c) in &triangles_of(n) {
        let ab = bits[e.iter().position(|&x| x == (a, b)).unwrap()];
        let ac = bits[e.iter().position(|&x| x == (a, c)).unwrap()];
        let bc = bits[e.iter().position(|&x| x == (b, c)).unwrap()];
        if ab == ac && ac == bc {
            return Some((a, b, c));
        }
    }
    None
}
fn count_avoiders(n: usize) -> (u64, Option<String>) {
    // enumeration exhaustive : nombre de colorations sans triangle monochrome + temoin
    let ne = edges_of(n).len();
    let mut cnt: u64 = 0;
    let mut witness: Option<String> = None;
    for_each_bits(ne, |bits| {
        if mono_triangle(n, bits).is_none() {
            cnt += 1;
            if witness.is_none() {
                witness = Some(bits.iter().map(|&b| b.to_string()).collect());
            }
        }
    });
    (cnt, witness)
}

let n = 4;
let (cnt, wit) = count_avoiders(n);
let ne = edges_of(n).len();
println!("K_{} : {} aretes, {} colorations", n, ne, 1u64 << ne);
println!("sans triangle monochrome : {} (exemple : {})", cnt, wit.unwrap());


K_4 : 6 aretes, 64 colorations


sans triangle monochrome : 18 (exemple : 001100)


### Lecture du résultat

Sur les $64$ colorations de $K_4$, dix-huit échappent au triangle monochrome — donc
$R(3,3) > 4$. Rien de surprenant : avec six arêtes, une coloration a beaucoup de
liberté. Ce premier palier fixe le geste : **le comptage exact est la preuve**, pas
une estimation. La question est de savoir jusqu'où il monte.

In [3]:
fn edges_of(n: usize) -> Vec<(usize, usize)> {
    let mut out = Vec::new();
    for i in 0..n { for j in i+1..n { out.push((i, j)); } }
    out
}
fn triangles_of(n: usize) -> Vec<(usize, usize, usize)> {
    let mut out = Vec::new();
    for a in 0..n { for b in a+1..n { for c in b+1..n { out.push((a,b,c)); } } }
    out
}
fn for_each_bits<F: FnMut(&[u8])>(m: usize, mut f: F) {
    for mask in 0u32..(1u32 << m) {
        let bits: Vec<u8> = (0..m).rev().map(|j| ((mask >> j) & 1) as u8).collect();
        f(&bits);
    }
}
fn mono_triangle(n: usize, bits: &[u8]) -> Option<(usize, usize, usize)> {
    let e = edges_of(n);
    for &(a, b, c) in &triangles_of(n) {
        let ab = bits[e.iter().position(|&x| x == (a, b)).unwrap()];
        let ac = bits[e.iter().position(|&x| x == (a, c)).unwrap()];
        let bc = bits[e.iter().position(|&x| x == (b, c)).unwrap()];
        if ab == ac && ac == bc {
            return Some((a, b, c));
        }
    }
    None
}
fn count_avoiders(n: usize) -> (u64, Option<String>) {
    let ne = edges_of(n).len();
    let mut cnt: u64 = 0;
    let mut witness: Option<String> = None;
    for_each_bits(ne, |bits| {
        if mono_triangle(n, bits).is_none() {
            cnt += 1;
            if witness.is_none() {
                witness = Some(bits.iter().map(|&b| b.to_string()).collect());
            }
        }
    });
    (cnt, witness)
}

let n = 5;
let (cnt, _wit) = count_avoiders(n);
let ne = edges_of(n).len();
println!("K_{} : {} aretes, {} colorations, sans triangle monochrome : {}", n, ne, 1u64 << ne, cnt);

// Structure des survivantes : le rouge est-il toujours un cycle C5 ?
let e = edges_of(n);
let mut n_c5: u64 = 0;
for_each_bits(ne, |bits| {
    if mono_triangle(n, bits).is_none() {
        let red: Vec<(usize, usize)> = e.iter().zip(bits.iter())
            .filter(|&(_, &b)| b == 1)
            .map(|(&ed, _)| ed)
            .collect();
        let mut deg = vec![0usize; n];
        for &(a, b) in &red {
            deg[a] += 1;
            deg[b] += 1;
        }
        if red.len() == 5 && deg == vec![2; n] {
            n_c5 += 1;
        }
    }
});
println!("survivantes dont le rouge est exactement un cycle C5 : {} / {}", n_c5, cnt);


K_5 : 10 aretes, 1024 colorations, sans triangle monochrome : 12


survivantes dont le rouge est exactement un cycle C5 : 12 / 12


### Lecture du résultat

Douze survivantes sur $1024$ : $R(3,3) > 5$. Et la structure est totale — la mesure le
dit : **chacune des 12 survivantes a pour rouge un cycle $C_5$ exactement** (cinq
arêtes, chaque sommet de degré 2), le bleu étant le $C_5$ complémentaire. Ce n'est pas
un hasard d'échantillon : 12 est précisément le nombre de cycles hamiltoniens étiquetés
de $K_5$, soit $\frac{5!}{2 \cdot 5} = 12$. Survivre à Ramsey sur $K_5$, c'est
**être** un double pentagone — la borne constructive de ce côté est un objet unique,
douze fois déguisé.

In [4]:
fn edges_of(n: usize) -> Vec<(usize, usize)> {
    let mut out = Vec::new();
    for i in 0..n { for j in i+1..n { out.push((i, j)); } }
    out
}
fn triangles_of(n: usize) -> Vec<(usize, usize, usize)> {
    let mut out = Vec::new();
    for a in 0..n { for b in a+1..n { for c in b+1..n { out.push((a,b,c)); } } }
    out
}
fn for_each_bits<F: FnMut(&[u8])>(m: usize, mut f: F) {
    for mask in 0u32..(1u32 << m) {
        let bits: Vec<u8> = (0..m).rev().map(|j| ((mask >> j) & 1) as u8).collect();
        f(&bits);
    }
}
fn mono_triangle(n: usize, bits: &[u8]) -> Option<(usize, usize, usize)> {
    let e = edges_of(n);
    for &(a, b, c) in &triangles_of(n) {
        let ab = bits[e.iter().position(|&x| x == (a, b)).unwrap()];
        let ac = bits[e.iter().position(|&x| x == (a, c)).unwrap()];
        let bc = bits[e.iter().position(|&x| x == (b, c)).unwrap()];
        if ab == ac && ac == bc {
            return Some((a, b, c));
        }
    }
    None
}
fn count_avoiders(n: usize) -> (u64, Option<String>) {
    let ne = edges_of(n).len();
    let mut cnt: u64 = 0;
    let mut witness: Option<String> = None;
    for_each_bits(ne, |bits| {
        if mono_triangle(n, bits).is_none() {
            cnt += 1;
            if witness.is_none() {
                witness = Some(bits.iter().map(|&b| b.to_string()).collect());
            }
        }
    });
    (cnt, witness)
}

let n = 6;
let (cnt, _) = count_avoiders(n);
let ne = edges_of(n).len();
println!("K_{} : {} aretes, {} colorations, sans triangle monochrome : {}", n, ne, 1u64 << ne, cnt);

// Ce que devient une coloration quelconque : ses triangles monochromes
let bits = [0u8, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0]; // 15 bits arbitraires
let e = edges_of(n);
let mut tris: Vec<(String, (usize, usize, usize))> = Vec::new();
for &(a, b, c) in &triangles_of(n) {
    let ab = bits[e.iter().position(|&x| x == (a, b)).unwrap()];
    let ac = bits[e.iter().position(|&x| x == (a, c)).unwrap()];
    let bc = bits[e.iter().position(|&x| x == (b, c)).unwrap()];
    if ab == ac && ac == bc {
        let couleur = if ab == 1 { "rouge" } else { "bleu" };
        tris.push((couleur.to_string(), (a, b, c)));
    }
}
let bits_str = format!("({})", bits.iter().map(|&b| b.to_string()).collect::<Vec<_>>().join(", "));
println!("\ncoloration arbitraire {} : {} triangles monochromes", bits_str, tris.len());
for (couleur, (a, b, c)) in tris.iter().take(4) {
    println!("  {} : ({}, {}, {})", couleur, a, b, c);
};

K_6 : 15 aretes, 32768 colorations, sans triangle monochrome : 0


coloration arbitraire (0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0) : 4 triangles monochromes


  bleu : (0, 1, 5)


  rouge : (0, 2, 3)


  bleu : (0, 4, 5)


  rouge : (1, 2, 4)


### Verdict — $R(3,3) = 6$

Zéro survivante sur $32768$ : **toute** coloration de $K_6$ porte un triangle
monochrome, et $K_5$ en évitait encore douze. L'énumération intégrale est une preuve
complète : $R(3,3) = 6$, exactement. Notons l'asymétrie du coût — $K_5$ a demandé
$1024$ vérifications, $K_6$ en demande $32768$ : un facteur 32 pour **un** sommet de
plus. Chaque palier multiplie le monde par $2^{n}$ environ. C'est la première fois que
le mur se montre, et il n'est pas encore haut.

## 2. Le pont comptage → existence : l'argument d'Erdős

Pour $R(4,4)$, la même preuve demanderait d'épuiser les colorations de $K_{18}$ (la
valeur exacte $R(4,4) = 18$ est connue — mais pas par épuisement naïf : la cellule
ci-dessous chiffre l'obstacle). L'argument probabiliste d'Erdős (1947) renverse la
question : pour montrer qu'une coloration **sans** $K_k$ monochrome existe, nul besoin
de la trouver — il suffit de montrer que les colorations qui en contiennent un sont
**moins nombreuses que le total**.

La forme générale : si $\binom{n}{k} \cdot 2^{1 - \binom{k}{2}} < 1$, alors
$R(k,k) > n$. La quantité de gauche majore le nombre **attendu** de copies
monochromes de $K_k$ dans une coloration aléatoire de $K_n$ ; si elle est inférieure
à 1, une coloration sans aucune copie existe forcément.


D'où vient la condition $\binom{n}{k} 2^{1-\binom{k}{2}} < 1$ ? Prenons une coloration aléatoire de $K_n$, chaque arête tirée rouge ou bleue avec probabilité $\frac{1}{2}$, indépendamment des autres. $K_n$ contient $\binom{n}{k}$ copies de $K_k$, et chacune est monochrome avec probabilité $2 \cdot 2^{-\binom{k}{2}} = 2^{1-\binom{k}{2}}$ (deux couleurs possibles, $\binom{k}{2}$ arêtes qui doivent toutes coïncider).

Par linéarité de l'espérance, le nombre espéré $X$ de copies monochromes de $K_k$ vaut $\binom{n}{k} 2^{1-\binom{k}{2}}$. Si cette espérance est strictement inférieure à 1, il existe forcément une coloration avec $X = 0$ — une variable aléatoire entière positive d'espérance $< 1$ prend la valeur 0 avec probabilité strictement positive — donc une coloration sans aucun $K_k$ monochrome, c'est-à-dire $R(k,k) > n$. C'est la **méthode du premier moment** : aucune hypothèse d'indépendance entre les événements « copie monochrome » n'est requise, seule compte la somme de leurs probabilités.

In [5]:
fn comb(n: u64, k: u64) -> u128 {
    if k > n { return 0; }
    let mut result = 1u128;
    for i in 0..k {
        result = result * ((n - i) as u128) / ((i + 1) as u128);
    }
    result
}
fn erdos_bound(k: u64) -> u64 {
    let mut n = 3u64;
    while (comb(n, k) as f64) * 2.0f64.powi(1i32 - comb(k, 2) as i32) < 1.0 {
        n += 1;
    }
    n - 1
}
fn py_exp(x: f64, precision: usize) -> String {
    let raw = format!("{:.*e}", precision, x);
    let (mantissa, exponent) = raw.split_once('e').unwrap();
    let value: i32 = exponent.parse().unwrap();
    format!("{}e{}{:02}", mantissa, if value >= 0 { "+" } else { "-" }, value.abs())
}

// Encadrements connus : survey dynamique DS1 de Radziszowski (Electronic J. Combinatorics)
let known: Vec<(u64, &str)> = vec![
    (3, "6 (exact)"), (4, "18 (exact)"), (5, "[43, 48]"),
    (6, "[102, 165]"), (7, "[205, 540]"), (8, "[281, 1870]"),
];

println!("{:>2} {:>13} {:>13} {:>12}", "k", "Erdos donne", "R(k,k) connu", "mur 2^C(k,2)");
for &(k, known_str) in &known {
    let n = erdos_bound(k);
    let wall = 2.0f64.powi(comb(k, 2) as i32);
    let wall_str = py_exp(wall, 2);
    println!("{:>2} {:>13} {:>13} {:>12}", k, format!("R > {}", n), known_str, wall_str);
};

 k   Erdos donne  R(k,k) connu mur 2^C(k,2)


 3         R > 3     6 (exact)     8.00e+00


 4         R > 6    18 (exact)     6.40e+01


 5        R > 11      [43, 48]     1.02e+03


 6        R > 17    [102, 165]     3.28e+04


 7        R > 27    [205, 540]     2.10e+06


 8        R > 42   [281, 1870]     2.68e+08


### Lecture du résultat

La borne donne $R(3,3) > 3$ — médiocre : l'épuisement de la section 1 a donné $> 5$.
Mais regardez la colonne du mur : le monde à énumérer double à chaque palier de $k$
croissant, et l'encadrement connu s'élargit en proportion ($[43, 48]$ pour $k = 5$ :
cinq valeurs possibles, personne ne sait trancher). La borne probabiliste est
**faible en petite dimension** — elle ne remplace pas l'énumération là où
l'énumération est possible — et c'est exactement pourquoi les deux coexistent : le
comptage exact règne jusqu'où il atteint, l'argument de comptage **majoré** règne
au-delà.

In [6]:
fn py_exp(x: f64, precision: usize) -> String {
    let raw = format!("{:.*e}", precision, x);
    let (mantissa, exponent) = raw.split_once('e').unwrap();
    let value: i32 = exponent.parse().unwrap();
    format!("{}e{}{:02}", mantissa, if value >= 0 { "+" } else { "-" }, value.abs())
}

// Le mur de R(4,4) : que couterait l'enumeration naive de K_18 ?
let colorings = 2.0f64.powi(153);
let rate = 1e9; // 1 milliard de colorations par seconde (optimiste)
let seconds = colorings / rate;
let years = seconds / (365.25 * 24.0 * 3600.0);
let age_universe = 1.38e10; // annees
println!("colorations de K_18 : 2^153 = {}", py_exp(colorings, 3));
println!("a 1e9 colorations/s : {} annees", py_exp(years, 2));
println!("age de l'univers    : {} annees", py_exp(age_universe, 2));
println!("rapport             : {} x l'age de l'univers", py_exp(years / age_universe, 1));


colorations de K_18 : 2^153 = 1.142e+46


a 1e9 colorations/s : 3.62e+29 annees


age de l'univers    : 1.38e+10 annees


rapport             : 2.6e+19 x l'age de l'univers


### Verdict — le pont comptage → existence

Même à un milliard de colorations par seconde, épuiser $K_{18}$ demande de l'ordre de
$10^{19}$ âges de l'univers (rapport exact imprimé ci-dessus) : le geste qui a prouvé
$R(3,3) = 6$ est **mort** à partir de là. L'inégalité d'Erdős — une ligne — donne
quant à elle $R(k,k) > n$ pour tout $n$ vérifiant la condition : c'est le pont. Ce que
l'énumération obtenait par constat exhaustif (« zéro survivante »), la probabilistique
l'obtient par majoration. La valeur exacte $R(4,4) = 18$ a été établie en 1955 par
Greenwood et Gleason — par une preuve combinatoire, pas par force brute.

## 3. Van der Waerden $W(2,3)$ : le même geste, un autre objet

**Énoncé.** Colorons chaque entier de $[1, n]$ en rouge ou bleu. $W(2,3)$ est le plus
petit $n$ tel que **toute** coloration contienne une progression arithmétique
monochrome de longueur 3 — trois entiers $a, a+d, a+2d$ de la même couleur. Le
comptage est le même qu'en section 1 ($2^n$ colorations à vérifier une à une) ; seul
le motif cherché change : la structure passe du graphe à la droite arithmétique.


Le théorème de van der Waerden (1927) généralise le constat : pour tout nombre de couleurs $r$ et toute longueur $k$, il existe un plus petit entier $W(r,k)$ tel que toute $r$-coloration de $[1, W(r,k)]$ contienne une progression arithmétique monochrome de longueur $k$. Ici $r = 2$, $k = 3$ : une 3-AP est un trio $a, a+d, a+2d$ — par exemple $1, 2, 3$ avec $d = 1$, ou $1, 5, 9$ avec $d = 4$.

Le moteur est strictement le même qu'en section 1 — énumérer les $2^m$ colorations de $[1..m]$ et tester un prédicat — seule la définition du prédicat change : `mono_ap` cherche une 3-AP monochrome là où `mono_triangle` cherchait un triangle. La cellule ci-dessous liste d'abord les colorations de $[1..8]$ qui évitent toute 3-AP monochrome ; la cellule suivante en fait le décompte pour chaque $m$ de 1 à 9.

In [7]:
fn mono_ap(m: usize, bits: &[u8]) -> Option<(usize, usize, usize)> {
    // premiere progression arithmetique monochrome de longueur 3 (None sinon)
    for a in 0..m {
        for d in 1..m {
            if a + 2 * d >= m {
                break;
            }
            if bits[a] == bits[a + d] && bits[a + d] == bits[a + 2 * d] {
                return Some((a, a + d, a + 2 * d));
            }
        }
    }
    None
}
fn for_each_bits<F: FnMut(&[u8])>(m: usize, mut f: F) {
    for mask in 0u32..(1u32 << m) {
        let bits: Vec<u8> = (0..m).rev().map(|j| ((mask >> j) & 1) as u8).collect();
        f(&bits);
    }
}

let m = 8;
let mut avoiders: Vec<String> = Vec::new();
for_each_bits(m, |bits| {
    if mono_ap(m, bits).is_none() {
        avoiders.push(bits.iter().map(|&b| b.to_string()).collect());
    }
});
println!("[1..{}] : {} colorations, sans 3-AP monochrome : {}", m, 1u64 << m, avoiders.len());
for w in &avoiders {
    println!("  {}", w);
};

[1..8] : 256 colorations, sans 3-AP monochrome : 6


  00110011


  01011010


  01100110


  10011001


  10100101


  11001100


### Lecture du résultat

Six survivantes sur $256$. Et la mesure dit mieux que le compte : **quatre** d'entre
elles sont des répétitions d'un motif de période 4 — `0011`, `0110` et leurs
compléments `1001`, `1100`. Les **deux** dernières, `01011010` et `10100101`
(compléments l'une de l'autre), ne sont **pas** périodiques : la contrainte laisse
donc passer aussi des formes irrégulières. C'est l'analogue du $C_5$ de la
section 1 : la contrainte ne laisse passer qu'un **petit nombre de formes** —
quatre régulières, deux irrégulières.

In [8]:
fn mono_ap(m: usize, bits: &[u8]) -> Option<(usize, usize, usize)> {
    for a in 0..m {
        for d in 1..m {
            if a + 2 * d >= m {
                break;
            }
            if bits[a] == bits[a + d] && bits[a + d] == bits[a + 2 * d] {
                return Some((a, a + d, a + 2 * d));
            }
        }
    }
    None
}
fn for_each_bits<F: FnMut(&[u8])>(m: usize, mut f: F) {
    for mask in 0u32..(1u32 << m) {
        let bits: Vec<u8> = (0..m).rev().map(|j| ((mask >> j) & 1) as u8).collect();
        f(&bits);
    }
}

// La chute : nombre d'evitantes pour chaque taille m = 1..9
for m in 1..=9usize {
    let mut count: u64 = 0;
    for_each_bits(m, |bits| {
        if mono_ap(m, bits).is_none() {
            count += 1;
        }
    });
    println!("m = {} : {:>4} colorations -> {:>3} evitantes", m, 1u64 << m, count);
};

m = 1 :    2 colorations ->   2 evitantes


m = 2 :    4 colorations ->   4 evitantes


m = 3 :    8 colorations ->   6 evitantes


m = 4 :   16 colorations ->  10 evitantes


m = 5 :   32 colorations ->  14 evitantes


m = 6 :   64 colorations ->  20 evitantes


m = 7 :  128 colorations ->  16 evitantes


m = 8 :  256 colorations ->   6 evitantes


m = 9 :  512 colorations ->   0 evitantes


### Lecture du résultat — la frontière exacte

La population croît d'abord (20 évitantes à $m = 6$), plafonne, puis s'effondre :
16, 6, et **zéro** à $m = 9$. Chaque entier ajouté apporte de nouvelles progressions
— donc de nouvelles contraintes — mais seulement un facteur 2 de liberté ; au-delà
d'un seuil, les contraintes gagnent. $W(2,3) = 9$ : la preuve est, comme pour
$R(3,3) = 6$, un épuisement — 512 colorations vérifiées une à une, aucune n'y
échappe. Deux mondes (graphes, arithmétique), la même méthode, deux certitudes
exactes.

## 4. Synthèse — deux certitudes exactes, un mur, un pont


Cette section ramasse en un seul tableau les cinq mesures du carnet : pour chaque monde — $K_4$, $K_5$, $K_6$, $[1..8]$, $[1..9]$ — le nombre total de colorations énumérées et le nombre de survivantes. Le coût se lit dans la colonne « taille », le rendement dans « survivantes » : c'est le contraste entre un coût qui double à chaque palier et un rendement qui s'effondre à zéro qui constitue la preuve.

In [9]:
// Recapitulatif mesure dans ce carnet
fn edges_of(n: usize) -> Vec<(usize, usize)> {
    let mut out = Vec::new();
    for i in 0..n { for j in i+1..n { out.push((i, j)); } }
    out
}
fn triangles_of(n: usize) -> Vec<(usize, usize, usize)> {
    let mut out = Vec::new();
    for a in 0..n { for b in a+1..n { for c in b+1..n { out.push((a,b,c)); } } }
    out
}
fn for_each_bits<F: FnMut(&[u8])>(m: usize, mut f: F) {
    for mask in 0u32..(1u32 << m) {
        let bits: Vec<u8> = (0..m).rev().map(|j| ((mask >> j) & 1) as u8).collect();
        f(&bits);
    }
}
fn mono_triangle(n: usize, bits: &[u8]) -> Option<(usize, usize, usize)> {
    let e = edges_of(n);
    for &(a, b, c) in &triangles_of(n) {
        let ab = bits[e.iter().position(|&x| x == (a, b)).unwrap()];
        let ac = bits[e.iter().position(|&x| x == (a, c)).unwrap()];
        let bc = bits[e.iter().position(|&x| x == (b, c)).unwrap()];
        if ab == ac && ac == bc {
            return Some((a, b, c));
        }
    }
    None
}
fn count_avoiders(n: usize) -> (u64, u64) {
    let ne = edges_of(n).len();
    let mut cnt: u64 = 0;
    for_each_bits(ne, |bits| {
        if mono_triangle(n, bits).is_none() {
            cnt += 1;
        }
    });
    (1u64 << ne, cnt)
}
fn mono_ap(m: usize, bits: &[u8]) -> Option<(usize, usize, usize)> {
    for a in 0..m {
        for d in 1..m {
            if a + 2 * d >= m {
                break;
            }
            if bits[a] == bits[a + d] && bits[a + d] == bits[a + 2 * d] {
                return Some((a, a + d, a + 2 * d));
            }
        }
    }
    None
}
fn count_ap_avoiders(m: usize) -> (u64, u64) {
    let mut cnt: u64 = 0;
    for_each_bits(m, |bits| {
        if mono_ap(m, bits).is_none() {
            cnt += 1;
        }
    });
    (1u64 << m, cnt)
}

let (t4, s4) = count_avoiders(4);
let (t5, s5) = count_avoiders(5);
let (t6, s6) = count_avoiders(6);
let (t8, s8) = count_ap_avoiders(8);
let (t9, s9) = count_ap_avoiders(9);

println!("{:<12} {:<8} {:>7}   survivantes", "resultat", "monde", "taille");
println!("{:<12} {:<8} {:>7}   {}", "R(3,3) > 4", "K_4", t4, s4);
println!("{:<12} {:<8} {:>7}   {}", "R(3,3) > 5", "K_5", t5, format!("{} (toutes des doubles C5)", s5));
println!("{:<12} {:<8} {:>7}   {}", "R(3,3) = 6", "K_6", t6, s6);
println!("{:<12} {:<8} {:>7}   {}", "W(2,3) > 8", "[1..8]", t8, format!("{} (4 periodiques + 2 irregulieres)", s8));
println!("{:<12} {:<8} {:>7}   {}", "W(2,3) = 9", "[1..9]", t9, s9);


resultat     monde     taille   survivantes


R(3,3) > 4   K_4           64   18


R(3,3) > 5   K_5         1024   12 (toutes des doubles C5)


R(3,3) = 6   K_6        32768   0


W(2,3) > 8   [1..8]       256   6 (4 periodiques + 2 irregulieres)


W(2,3) = 9   [1..9]       512   0


### Verdict de synthèse

L'énumération a livré deux théorèmes ($R(3,3) = 6$, $W(2,3) = 9$) et des bornes
constructives en témoin ($R(3,3) > 5$, $W(2,3) > 8$) — à coût dérisoire. Le même
geste sur $R(4,4)$ se heurte à $2^{153}$ : le mur. Le pont d'Erdős remplace « toutes
vérifiées » par « moins nombreuses que le tout », et l'existence repose sur une
inégalité au lieu d'un constat. C'est la leçon de structure : **le comptage exact et
la majoration probabiliste ne sont pas deux méthodes concurrentes mais deux régimes du
même geste** — l'un règne tant que le monde tient en mémoire, l'autre prend le relais
et ne rend plus la valeur exacte, seulement un encadrement ($43 \le R(5,5) \le 48$ à
ce jour, survey DS1 de Radziszowski).

## Exercices

Trois exercices, tous sur le même moteur que les sections 1 à 3. Les stubs s'exécutent
sans erreur ; à vous de remplir le calcul.


Chaque exercice réutilise les fonctions définies dans les sections 1 à 3, sans introduire de nouveau concept : le moteur d'énumération est le même, seul le motif cherché change — un cycle $C_4$ rouge parmi les survivantes de $K_4$, un trio monochrome $x+y=z$ (Schur), ou la borne d'Erdős prolongée au-delà de $k = 8$.

### Exercice 1 — $K_4$ : les 18 survivantes, et leurs cycles

La section 1 a compté 18 colorations de $K_4$ sans triangle monochrome. Parmi elles,
combien ont pour rouge exactement un cycle $C_4$ (quatre arêtes rouges, chaque sommet
de degré rouge 2) ? Vérifiez votre réponse contre la valeur théorique : le nombre de
cycles de longueur 4 étiquetés sur 4 sommets.

In [10]:
// Exercice 1 -- parmi les 18 evitantes de K_4, compter les rouges = C4 (a completer)
// Etape 1 : reprendre l'enumeration de count_avoiders(4) en gardant la liste des colorations.
// Etape 2 : pour chaque survivante, reconstruire les aretes rouges et les degres.
// Indice : la reponse theorique est (4-1)!/2 = 3 cycles etiquetes.
let _resultat: Option<u64> = None; // TODO etudiant
println!("Exercice a completer");


Exercice a completer


### Exercice 2 — Schur : le troisième monde, $S(2) = 5$

Colorons $[1, n]$ en rouge et bleu ; une solution **monochrome** de $x + y = z$ est un
trio d'une même couleur. Le nombre de Schur $S(2) = 5$ : $[1..4]$ est coloriable sans
trio monochrome, $[1..5]$ ne l'est pas. Même moteur d'énumération, troisième structure
— l'addition.

In [11]:
// Exercice 2 -- enumeration des colorations de [1..m] sans trio monochrome x+y=z (a completer)
// Etape 1 : lister les solutions (x, y, z) avec x + y = z dans [1..m] (attention : 1+2 et 2+1
//            sont deux solutions distinctes, mais une seule contrainte).
// Etape 2 : enumerer les 2^m colorations, garder celles sans trio monochrome.
// Indice : il en reste 2 sur [1..4], et 0 sur [1..5].
fn schur_avoiders(_m: usize) -> Option<Vec<Vec<u8>>> {
    // TODO etudiant : renvoyer la liste des colorations (tuples de bits) sans trio mono
    None
}
let _ = schur_avoiders;
println!("Exercice a completer");


Exercice a completer


### Exercice 3 — Jusqu'où l'inégalité d'Erdős grimpe-t-elle ?

La section 2 s'arrête à $k = 8$. Prolongez la table : $k = 9$ à $12$, borne d'Erdős
d'un côté ; de l'autre, l'encadrement courant de $R(k,k)$ tel que publié dans la
survey dynamique DS1 de Radziszowski (*Small Ramsey Numbers*, Electronic Journal of
Combinatorics — allez le chercher : c'est l'exercice de lecture). Que fait le
**rapport** entre la borne d'Erdős et la meilleure borne inférieure connue quand $k$
croît ?

In [12]:
// Exercice 3 -- table d'Erdos pour k = 9..12 (a completer)
// Etape 1 : reprendre la boucle de la section 2 avec range(9, 13).
// Etape 2 : ajouter log2 de la borne -- la croissance est-elle lineaire en k ?
// Indice : la borne croit comme 2^{k/2} environ ; comparez au mur 2^{C(k,2)}.
let _bornes: Option<Vec<(u64, u64, f64)>> = None; // TODO etudiant
println!("Exercice a completer");


Exercice a completer


## Conclusion

Ce carnet a prouvé deux théorèmes en comptant tout, mesuré le mur où ce geste meurt
($2^{153}$ colorations pour $K_{18}$), et traversé ce mur par l'inégalité d'Erdős.
Trois structures — triangle de graphe, progression arithmétique, équation additive —
un seul moteur : énumérer, compter, conclure.

Références :

- **Erdős, P. (1947).** *Some remarks on the theory of graphs.* Bulletin of the
  American Mathematical Society. — l'argument probabiliste fondateur.
- **Greenwood, R. E. & Gleason, A. M. (1955).** *Combinatorial relations and
  chromatic graphs.* Canadian Journal of Mathematics. — $R(3,3) = 6$, $R(4,4) = 18$
  établis combinatoirement.
- **Radziszowski, S.** *Small Ramsey Numbers.* Electronic Journal of Combinatorics,
  Dynamic Survey DS1 (révisions successives). — l'état courant des encadrements.
- **van der Waerden, B. L. (1927).** *Beweis einer Baudetschen Vermutung.* Nieuw
  Archief voor Wiskunde. — le théorème, dans sa formulation d'origine.

> Carnet issu de l'EPIC #19898.